<a href="https://colab.research.google.com/github/logonzalezsa/PPAE/blob/main/02_Cuadernos/PPAE_02_CuantiaRefuerzo_Flexion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROGRAMACIÓN PARA ANÁLISIS DE ESTRUCTURAS

**Universidad Nacional de Colombia – Sede Palmira**  
**Programa Curricular de Ingeniería Agrícola**

### Tema
**Tópicos sobre Programación de Computadores**

### Taller
**Aplicaciones en Ingeniería Estructural**

### Ejercicio
**Cuantía de Refuerzo en Elementos de Hormigón Sometidos a Flexión (vigas)**

**Elaborado por:**  
Luis Octavio González Salcedo  
Profesor Titular

## 1. Contexto ingenieril

Las vigas de hormigón reforzado están sometidas principalmente a esfuerzos de flexión producidos por las acciones que actúan sobre la estructura. Como consecuencia, se generan zonas sometidas a compresión y zonas sometidas a tracción, cuya ubicación depende del signo del momento flector.

El hormigón presenta una elevada capacidad para resistir esfuerzos de compresión, mientras que el refuerzo de acero se dispone para contribuir principalmente a resistir los esfuerzos de tracción. Por esta razón, el diseño a flexión requiere determinar el área de acero necesaria, $A_s$, asociada con los momentos de diseño.

En este cuaderno se considerarán dos valores de momento:

- Momento máximo positivo, $M_u^+$.
- Momento máximo negativo, $M_u^-$.

Para orientar inicialmente las dimensiones de la viga se utilizará el mayor momento en valor absoluto y una **cuantía de predimensionamiento**, $\rho_{pre}$. A partir de esta información se obtendrá una sección de referencia.

Posteriormente, el diseñador adoptará las dimensiones constructivas de la sección, $b$ y $h$, y el programa determinará, para cada momento, la cuantía de refuerzo requerida y su correspondiente área de acero:

$$
\rho^+ \longrightarrow A_s^+
$$

$$
\rho^- \longrightarrow A_s^-
$$

La ubicación del refuerzo correspondiente a cada área deberá establecerse a partir del signo del momento flector y de la identificación de la zona sometida a tracción.

### Objetivo del cuaderno

Implementar un procedimiento computacional para predimensionar una sección rectangular de viga de hormigón reforzado y determinar las cuantías y áreas de refuerzo requeridas para sus momentos máximos positivo y negativo, verificando posteriormente la conveniencia de la sección adoptada.

## 2. Formulación del problema

Para una sección rectangular de hormigón reforzado sometida a flexión, se utilizará la siguiente expresión para relacionar el momento último de diseño, la geometría de la sección y la cuantía de refuerzo:

$$
\frac{M_u}{bd^2}
=
0.90\,\rho f_y
\left(
1-0.59\frac{\rho f_y}{f'_c}
\right)
$$

donde:

- $M_u$: momento último de diseño [N·mm].
- $b$: ancho de la sección [mm].
- $d$: altura efectiva de la sección [mm].
- $f'_c$: resistencia especificada a la compresión del hormigón [MPa].
- $f_y$: esfuerzo de fluencia del acero de refuerzo [MPa].
- $\rho$: cuantía de refuerzo longitudinal.

La cuantía de refuerzo se define como:

$$
\rho=\frac{A_s}{bd}
$$

por lo tanto:

$$
A_s=\rho bd
$$

donde $A_s$ corresponde al área de acero de refuerzo [mm²].

Para este ejercicio se utilizarán como valores de referencia:

$$
\rho_{min}=0.0033
$$

$$
\rho_{max}=0.0160
$$

El procedimiento se desarrollará en dos etapas:

**Predimensionamiento de la sección**

Se utilizará el mayor valor absoluto entre los momentos máximo positivo y máximo negativo:

$$
M_{u,\mathrm{control}}
=
\max\left(|M_u^+|,\;|M_u^-|\right)
$$

y una cuantía de predimensionamiento, $\rho_{pre}$, para obtener una sección rectangular de referencia.

**Diseño de la sección adoptada**

Una vez establecidas las dimensiones constructivas $b$ y $h$, se calculará:

$$
d=h-d'
$$

donde $d'$ representa la distancia desde la cara extrema de la sección hasta el centro del refuerzo longitudinal considerado.

Con la sección adoptada se determinarán independientemente las cuantías requeridas para los momentos positivo y negativo:

$$
\rho^+ \longrightarrow A_s^+
$$

$$
\rho^- \longrightarrow A_s^-
$$

Finalmente, las cuantías obtenidas se compararán con los valores de referencia establecidos para evaluar la conveniencia de la sección adoptada.

## 3. Datos de entrada y sistema de unidades

Para desarrollar el procedimiento se utilizarán los siguientes datos de entrada:

- Momento máximo positivo, $M_u^+$ [kN·m].
- Momento máximo negativo, $M_u^-$ [kN·m].
- Resistencia especificada a la compresión del hormigón, $f'_c$ [MPa].
- Esfuerzo de fluencia del acero de refuerzo, $f_y$ [MPa].
- Distancia desde la cara extrema de la sección hasta el centro del refuerzo longitudinal, $d'$ [mm].
- Cuantía adoptada para el predimensionamiento, $\rho_{pre}$ [adimensional].

### Sistema de unidades

Los datos serán ingresados utilizando:

$$
M_u \; [\mathrm{kN\cdot m}]
$$

$$
f'_c,\;f_y \; [\mathrm{MPa}]
$$

$$
b,\;h,\;d,\;d' \; [\mathrm{mm}]
$$

Para realizar los cálculos, los momentos se convertirán internamente de kN·m a N·mm mediante:

$$
1\;\mathrm{kN\cdot m}=10^6\;\mathrm{N\cdot mm}
$$

Recordando que:

$$
1\;\mathrm{MPa}=1\;\mathrm{N/mm^2}
$$

el sistema de unidades utilizado permite obtener directamente el área de refuerzo, $A_s$, en mm².

En una etapa posterior del procedimiento, el usuario deberá ingresar las dimensiones de la sección que desea evaluar:

- Ancho adoptado de la sección, $b$ [mm].
- Altura total adoptada de la sección, $h$ [mm].

Estas dimensiones podrán ser diferentes de las obtenidas durante el predimensionamiento, ya que este último constituye una referencia para orientar la selección de una sección constructiva.

## 4. Implementación computacional

### 4.1. Importación de librerías

Para desarrollar el procedimiento se utilizará la biblioteca **NumPy**, que proporciona herramientas para realizar operaciones numéricas y algebraicas.

En particular, se utilizará para efectuar operaciones matemáticas asociadas con el predimensionamiento y con la solución de la ecuación de segundo grado que permite determinar la cuantía de refuerzo.

In [ ]:
import numpy as np

### 4.2. Datos de entrada

En este bloque se ingresan los datos necesarios para realizar el predimensionamiento de la viga y, posteriormente, calcular las cuantías de refuerzo correspondientes a los momentos máximo positivo y máximo negativo.

Los momentos se ingresarán en kN·m, las resistencias en MPa, las dimensiones en mm y la cuantía de predimensionamiento en forma decimal.

In [ ]:
# ============================================================
# DATOS DE ENTRADA
# ============================================================

Mu_pos = float(input("Momento máximo positivo Mu+ [kN·m]: "))
Mu_neg = float(input("Momento máximo negativo Mu- [kN·m]: "))

fc = float(input("Resistencia especificada del hormigón f'c [MPa]: "))
fy = float(input("Esfuerzo de fluencia del acero fy [MPa]: "))

d_prima = float(input("Distancia d' [mm]: "))

rho_pre = float(
    input("Cuantía de predimensionamiento rho_pre [decimal, por ejemplo 0.010]: ")
)

### 4.3. Momento de control para el predimensionamiento

Para orientar el predimensionamiento de la sección se utilizará el mayor valor absoluto entre el momento máximo positivo y el momento máximo negativo.

$$
M_{u,\mathrm{control}}
=
\max\left(|M_u^+|,\;|M_u^-|\right)
$$

El signo del momento no interviene en esta etapa, ya que el objetivo es establecer una geometría de referencia para la viga. Posteriormente, una vez adoptada la sección, los momentos positivo y negativo serán considerados independientemente para determinar sus correspondientes cuantías y áreas de refuerzo.

Para mantener la consistencia del sistema de unidades, el momento que controla el predimensionamiento se convertirá de kN·m a N·mm.

In [ ]:
# ============================================================
# MOMENTO DE CONTROL PARA EL PREDIMENSIONAMIENTO
# ============================================================

Mu_control_kNm = max(abs(Mu_pos), abs(Mu_neg))

# Conversión de kN·m a N·mm
Mu_control = Mu_control_kNm * 1e6

print("Momento máximo positivo |Mu+| =", abs(Mu_pos), "kN·m")
print("Momento máximo negativo |Mu-| =", abs(Mu_neg), "kN·m")
print()
print("Momento que controla el predimensionamiento =", Mu_control_kNm, "kN·m")
print("Momento de control para el cálculo =", Mu_control, "N·mm")

### 4.4. Predimensionamiento de la sección

A partir del momento que controla el predimensionamiento y de la cuantía adoptada, $\rho_{pre}$, se utiliza la expresión:

$$
\frac{M_{u,\mathrm{control}}}{bd^2}
=
0.90\,\rho_{pre} f_y
\left(
1-0.59\frac{\rho_{pre}f_y}{f'_c}
\right)
$$

Despejando el parámetro geométrico $bd^2$:

$$
bd^2=
\frac{M_{u,\mathrm{control}}}
{0.90\,\rho_{pre}f_y
\left(
1-0.59\frac{\rho_{pre}f_y}{f'_c}
\right)}
$$

Para obtener unas dimensiones iniciales de la sección se adopta, como criterio de predimensionamiento:

$$
d=2b
$$

Por lo tanto:

$$
bd^2=b(2b)^2=4b^3
$$

y:

$$
b=\sqrt[3]{\frac{bd^2}{4}}
$$

Una vez determinado $b$, se calcula:

$$
d=2b
$$

y la altura total teórica de la sección será:

$$
h=d+d'
$$

Los valores obtenidos constituyen dimensiones teóricas de referencia. Posteriormente se ajustarán a dimensiones constructivas.

In [ ]:
# ============================================================
# PREDIMENSIONAMIENTO DE LA SECCIÓN
# ============================================================

# Cálculo del parámetro geométrico requerido bd²
bd2 = Mu_control / (
    0.90 * rho_pre * fy *
    (1 - 0.59 * rho_pre * fy / fc)
)

# Dimensiones teóricas considerando d = 2b
b_teorico = np.cbrt(bd2 / 4)
d_teorico = 2 * b_teorico
h_teorico = d_teorico + d_prima

print("PREDIMENSIONAMIENTO TEÓRICO")
print("---------------------------")
print(f"bd² requerido = {bd2:.2f} mm³")
print(f"b teórico     = {b_teorico:.2f} mm")
print(f"d teórico     = {d_teorico:.2f} mm")
print(f"h teórico     = {h_teorico:.2f} mm")

### 4.5. Ajuste a dimensiones constructivas

Las dimensiones obtenidas durante el predimensionamiento son valores teóricos. Para proponer una sección de referencia con dimensiones constructivas, se ajustarán el ancho y la altura total al múltiplo superior de 25 mm.

Este ajuste proporciona una **sección sugerida para orientar al diseñador**, pero no constituye necesariamente la sección definitiva que será utilizada en el cálculo del refuerzo.

Una vez ajustada la altura total, la altura efectiva correspondiente se obtiene mediante:

$$
d_{sug}=h_{sug}-d'
$$

In [ ]:
# ============================================================
# AJUSTE A DIMENSIONES CONSTRUCTIVAS
# ============================================================

multiplo = 25  # mm

b_sugerido = np.ceil(b_teorico / multiplo) * multiplo
h_sugerido = np.ceil(h_teorico / multiplo) * multiplo
d_sugerido = h_sugerido - d_prima

print("SECCIÓN SUGERIDA PARA EL PREDIMENSIONAMIENTO")
print("--------------------------------------------")
print(f"b sugerido = {b_sugerido:.0f} mm")
print(f"h sugerido = {h_sugerido:.0f} mm")
print(f"d sugerido = {d_sugerido:.0f} mm")

### 4.6. Sección adoptada y formulación de la cuantía de refuerzo

El predimensionamiento proporciona una sección de referencia para orientar al diseñador. Sin embargo, la sección utilizada finalmente en el diseño puede ser diferente, considerando criterios estructurales, constructivos, arquitectónicos o de coordinación con otros elementos del proyecto.

Por esta razón, el usuario deberá adoptar las dimensiones:

- $b$: ancho de la sección [mm].
- $h$: altura total de la sección [mm].

La altura efectiva se determinará mediante:

$$
d=h-d'
$$

Para la sección adoptada, la cuantía de refuerzo requerida se obtiene a partir de:

$$
\frac{M_u}{bd^2}
=
0.90\,\rho f_y
\left(
1-0.59\frac{\rho f_y}{f'_c}
\right)
$$

Desarrollando la expresión:

$$
\frac{M_u}{bd^2}
=
0.90\rho f_y
-
0.531\frac{f_y^2}{f'_c}\rho^2
$$

y reorganizando:

$$
0.531\frac{f_y^2}{f'_c}\rho^2
-
0.90f_y\rho
+
\frac{M_u}{bd^2}
=
0
$$

Esta expresión corresponde a una ecuación de segundo grado:

$$
a\rho^2+b\rho+c=0
$$

donde:

$$
a=0.531\frac{f_y^2}{f'_c}
$$

$$
b=-0.90f_y
$$

$$
c=\frac{M_u}{bd^2}
$$

Como los momentos positivo y negativo son diferentes, se obtendrán dos valores del término independiente:

$$
c^+=\frac{|M_u^+|}{bd^2}
$$

$$
c^-=\frac{|M_u^-|}{bd^2}
$$

### 4.7. Solución matemática y verificación de la sección

Para cada momento se resolverá la correspondiente ecuación de segundo grado. El discriminante se define como:

$$
\Delta=b^2-4ac
$$

Si:

$$
\Delta\geq0
$$

existen soluciones reales y las raíces se obtienen mediante:

$$
\rho_1=\frac{-b-\sqrt{\Delta}}{2a}
$$

$$
\rho_2=\frac{-b+\sqrt{\Delta}}{2a}
$$

El programa mostrará las dos raíces obtenidas para cada momento. Sin embargo, una solución matemática no constituye necesariamente una solución admisible desde el punto de vista de la ingeniería.

Para este ejercicio se utilizará la raíz menor correspondiente a cada momento y se verificará su valor con respecto a la cuantía máxima de referencia:

$$
\rho_{max}=0.0160
$$

Si alguna de las cuantías requeridas supera este valor, la sección adoptada será considerada inadecuada y deberá proponerse una nueva sección.

El procedimiento se repetirá hasta encontrar una sección que satisfaga el criterio establecido.

Adicionalmente, cuando alguna cuantía se encuentre próxima al límite máximo, el programa generará una advertencia para que el diseñador evalúe la conveniencia de aumentar las dimensiones de la sección.



In [ ]:
# ============================================================
# SECCIÓN ADOPTADA, CÁLCULO Y VERIFICACIÓN DE CUANTÍAS
# ============================================================

rho_min = 0.0033
rho_max = 0.0160
rho_advertencia = 0.90 * rho_max

# Coeficientes de la ecuación cuadrática que dependen
# únicamente de las propiedades de los materiales
a = 0.531 * fy**2 / fc
b_coef = -0.90 * fy

# Conversión de los momentos a N·mm
Mu_pos_Nmm = abs(Mu_pos) * 1e6
Mu_neg_Nmm = abs(Mu_neg) * 1e6

seccion_aceptable = False

while not seccion_aceptable:

    print("\nSECCIÓN SUGERIDA POR EL PREDIMENSIONAMIENTO")
    print("--------------------------------------------")
    print(f"b sugerido = {b_sugerido:.0f} mm")
    print(f"h sugerido = {h_sugerido:.0f} mm")

    print("\nIngrese la sección que desea evaluar:")

    b = float(input("Ancho adoptado b [mm]: "))
    h = float(input("Altura total adoptada h [mm]: "))

    d = h - d_prima

    # Términos independientes de las ecuaciones
    c_pos = Mu_pos_Nmm / (b * d**2)
    c_neg = Mu_neg_Nmm / (b * d**2)

    # Discriminantes
    delta_pos = b_coef**2 - 4 * a * c_pos
    delta_neg = b_coef**2 - 4 * a * c_neg

    print("\nRESULTADOS PARA LA SECCIÓN ADOPTADA")
    print("-----------------------------------")
    print(f"b  = {b:.0f} mm")
    print(f"h  = {h:.0f} mm")
    print(f"d' = {d_prima:.0f} mm")
    print(f"d  = {d:.0f} mm")

    # Primero se verifica la existencia de raíces reales
    if delta_pos < 0 or delta_neg < 0:

        print("\nLa sección adoptada no permite obtener")
        print("una solución real para ambos momentos.")
        print("Debe seleccionar una nueva sección.")

    else:

        # Dos raíces para cada momento
        rho_pos_1 = (-b_coef - np.sqrt(delta_pos)) / (2 * a)
        rho_pos_2 = (-b_coef + np.sqrt(delta_pos)) / (2 * a)

        rho_neg_1 = (-b_coef - np.sqrt(delta_neg)) / (2 * a)
        rho_neg_2 = (-b_coef + np.sqrt(delta_neg)) / (2 * a)

        print("\nRAÍCES OBTENIDAS")
        print("----------------")
        print(f"ρ1+ = {rho_pos_1:.6f}")
        print(f"ρ2+ = {rho_pos_2:.6f}")
        print(f"ρ1- = {rho_neg_1:.6f}")
        print(f"ρ2- = {rho_neg_2:.6f}")

        # Se adopta la raíz menor
        rho_pos = rho_pos_1
        rho_neg = rho_neg_1

        print("\nCUANTÍAS REQUERIDAS")
        print("-------------------")
        print(f"ρ+ = {rho_pos:.6f}")
        print(f"ρ- = {rho_neg:.6f}")

        # Verificación del límite máximo
        if rho_pos > rho_max or rho_neg > rho_max:

            print("\nSECCIÓN NO ACEPTABLE")
            print(f"Alguna cuantía supera ρmax = {rho_max:.4f}.")
            print("Debe seleccionar una nueva sección.")

        else:

            seccion_aceptable = True

            # Advertencia si alguna cuantía está próxima al máximo
            if rho_pos >= rho_advertencia or rho_neg >= rho_advertencia:

                print("\nADVERTENCIA")
                print(
                    f"Alguna cuantía es igual o superior a "
                    f"{rho_advertencia:.4f} (90 % de ρmax)."
                )
                print("La sección cumple el límite establecido,")
                print("pero se recomienda evaluar una sección mayor.")

            else:

                print("\nLa sección adoptada resulta aceptable")
                print("respecto al límite máximo de cuantía.")

### 4.8. Determinación de las áreas de refuerzo

Una vez aceptada la sección y obtenidas las cuantías requeridas para los momentos positivo y negativo, se verifica el cumplimiento de la cuantía mínima establecida para este ejercicio:

$$
\rho_{min}=0.0033
$$

Si la cuantía calculada para alguno de los momentos es inferior a $\rho_{min}$, para el cálculo del área de refuerzo se adoptará la cuantía mínima.

Por lo tanto:

$$
\rho_{adoptada}=
\max(\rho_{calculada},\rho_{min})
$$

El área de refuerzo correspondiente se determina mediante:

$$
A_s=\rho_{adoptada}bd
$$

Para los momentos positivo y negativo:

$$
A_s^+=\rho_{adoptada}^+bd
$$

$$
A_s^-=\rho_{adoptada}^-bd
$$

El programa reportará separadamente las áreas de refuerzo asociadas con cada momento. La selección del número y diámetro de las barras, así como su ubicación en la sección transversal, corresponde a una decisión posterior de diseño.

In [ ]:
# ============================================================
# DETERMINACIÓN DE LAS ÁREAS DE REFUERZO
# ============================================================

# Verificación de la cuantía mínima
rho_pos_adoptada = max(rho_pos, rho_min)
rho_neg_adoptada = max(rho_neg, rho_min)

# Cálculo de las áreas de refuerzo
As_pos = rho_pos_adoptada * b * d
As_neg = rho_neg_adoptada * b * d

print("ÁREAS DE REFUERZO")
print("-----------------")

print("\nMomento positivo:")
print(f"ρ calculada = {rho_pos:.6f}")

if rho_pos < rho_min:
    print(f"ρ calculada < ρmin = {rho_min:.4f}")
    print(f"Se adopta ρmin = {rho_pos_adoptada:.4f}")
else:
    print(f"ρ adoptada  = {rho_pos_adoptada:.6f}")

print(f"As+ = {As_pos:.2f} mm²")

print("\nMomento negativo:")
print(f"ρ calculada = {rho_neg:.6f}")

if rho_neg < rho_min:
    print(f"ρ calculada < ρmin = {rho_min:.4f}")
    print(f"Se adopta ρmin = {rho_neg_adoptada:.4f}")
else:
    print(f"ρ adoptada  = {rho_neg_adoptada:.6f}")

print(f"As- = {As_neg:.2f} mm²")

### 4.9. Resumen final del procedimiento

El siguiente resumen integra los principales resultados obtenidos durante el procedimiento computacional.

Se presentan el momento que controló el predimensionamiento, la sección sugerida inicialmente, la sección finalmente adoptada por el diseñador y las cuantías y áreas de refuerzo correspondientes a los momentos positivo y negativo.

La comparación entre la sección sugerida y la sección adoptada permite observar que el predimensionamiento constituye una herramienta de orientación y no sustituye la decisión del diseñador.

De igual manera, las áreas de refuerzo obtenidas representan la demanda de acero asociada con cada momento. A partir de estos resultados deberá definirse posteriormente una disposición constructiva de barras que satisfaga el área requerida y sea coherente con el comportamiento estructural del elemento.

In [ ]:
# ============================================================
# RESUMEN FINAL DEL PROCEDIMIENTO
# ============================================================

print("=" * 58)
print("RESUMEN FINAL - DISEÑO A FLEXIÓN DE LA VIGA")
print("=" * 58)

print("\nDATOS PRINCIPALES")
print("-----------------")
print(f"Mu+ = {Mu_pos:.2f} kN·m")
print(f"Mu- = {Mu_neg:.2f} kN·m")
print(f"Mu de control = {Mu_control_kNm:.2f} kN·m")
print(f"f'c = {fc:.2f} MPa")
print(f"fy  = {fy:.2f} MPa")
print(f"d'  = {d_prima:.2f} mm")
print(f"ρpre = {rho_pre:.5f}")

print("\nPREDIMENSIONAMIENTO")
print("-------------------")
print(f"b teórico  = {b_teorico:.2f} mm")
print(f"h teórico  = {h_teorico:.2f} mm")
print(f"b sugerido = {b_sugerido:.0f} mm")
print(f"h sugerido = {h_sugerido:.0f} mm")

print("\nSECCIÓN ADOPTADA")
print("----------------")
print(f"b = {b:.0f} mm")
print(f"h = {h:.0f} mm")
print(f"d = {d:.0f} mm")

print("\nREFUERZO ASOCIADO AL MOMENTO POSITIVO")
print("-------------------------------------")
print(f"ρ+ calculada = {rho_pos:.6f}")
print(f"ρ+ adoptada  = {rho_pos_adoptada:.6f}")
print(f"As+           = {As_pos:.2f} mm²")

print("\nREFUERZO ASOCIADO AL MOMENTO NEGATIVO")
print("-------------------------------------")
print(f"ρ- calculada = {rho_neg:.6f}")
print(f"ρ- adoptada  = {rho_neg_adoptada:.6f}")
print(f"As-           = {As_neg:.2f} mm²")

print("\nLÍMITES DE REFERENCIA")
print("---------------------")
print(f"ρmin = {rho_min:.4f}")
print(f"ρmax = {rho_max:.4f}")

print("\n" + "=" * 58)
print("INTERPRETACIÓN INGENIERIL")
print("=" * 58)
print("La sección adoptada satisface el criterio máximo de")
print("cuantía utilizado en este ejercicio.")
print()
print("Las áreas As+ y As- corresponden al refuerzo requerido")
print("para los momentos positivo y negativo, respectivamente.")
print()
print("El diseñador deberá definir el número, diámetro y")
print("ubicación de las barras de refuerzo.")

## Reflexión final

El procedimiento computacional permite realizar los cálculos de manera sistemática y verificar diferentes alternativas de sección. Sin embargo, la solución numérica no completa por sí sola el diseño estructural.

A partir de los resultados obtenidos, analice:

1. ¿Qué momento controló el predimensionamiento de la sección?
2. ¿Por qué la sección adoptada puede ser diferente de la sección sugerida?
3. ¿Dónde debe ubicarse el refuerzo asociado con $A_s^+$?
4. ¿Dónde debe ubicarse el refuerzo asociado con $A_s^-$?
5. Seleccione una combinación de barras comerciales que satisfaga cada área de acero requerida.
6. ¿Qué decisión tomaría si una de las cuantías calculadas estuviera próxima a $\rho_{max}$?